In [ ]:
!nvidia-smi

Mon Aug 10 01:26:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%cd /content
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5
!git checkout v6.2
!pip install -r requirements.txt


/content
Cloning into 'yolov5'...
remote: Enumerating objects: 18689, done.
remote: Counting objects: 100% (291/291), done.
remote: Compressing objects: 100% (166/166), done.
remote: Total 18689 (delta 240), reused 125 (delta 125), pack-reused 18398 (from 3)
Receiving objects: 100% (18689/18689), 17.68 MiB | 9.23 MiB/s, done.
Resolving deltas: 100% (12731/12731), done.
/content/yolov5
Note: switching to 'v6.2'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at d3ea0df8 New YOLOv5 Classification Models (#8956)
   

In [ ]:
import os
print(os.path.exists("/content/yolov5"))
print(os.path.exists("/content/yolov5/train.py"))

True
True


In [ ]:
!sed -i 's/np\.int)/int)/g' utils/dataloaders.py #numpy 패치

In [ ]:
# corrosion 데이터셋 전용 강화 augmentation 하이퍼파라미터
# 기존 data/hyps/hyp.scratch-low.yaml 대비 변경점:
# - 회전/원근/이동 폭 확대 (카메라 각도 다양성 대응)
# - 밝기/채도 변화 폭 확대 (조명 환경 다양성 대응)
# - mixup 활성화 (데이터가 적을 때 유효 다양성 증가)
# - scale 폭 확대 (거리/줌 다양성 대응)

lr0: 0.01
lrf: 0.01
momentum: 0.937
weight_decay: 0.0005
warmup_epochs: 3.0
warmup_momentum: 0.8
warmup_bias_lr: 0.1
box: 0.05
cls: 0.5
cls_pw: 1.0
obj: 1.0
obj_pw: 1.0
iou_t: 0.2
anchor_t: 4.0
fl_gamma: 0.0

# ===== 여기부터 augmentation 관련, 기본값보다 강화 =====
hsv_h: 0.02      # 기본 0.015 -> 색조 변화 폭 소폭 확대
hsv_s: 0.8       # 기본 0.7 -> 채도 변화 강화 (녹/부식 색상 다양성 대응)
hsv_v: 0.5       # 기본 0.4 -> 명도(밝기) 변화 강화 (조명 환경 대응)
degrees: 15.0    # 기본 0.0 -> 회전 각도 부여 (카메라 각도 다양성)
translate: 0.15  # 기본 0.1 -> 이동 폭 확대
scale: 0.7       # 기본 0.5 -> 확대/축소 폭 확대 (촬영 거리 다양성)
shear: 5.0       # 기본 0.0 -> 약간의 전단 변형 추가
perspective: 0.0005  # 기본 0.0 -> 원근 왜곡 소량 추가 (카메라 각도 대응)
flipud: 0.3      # 기본 0.0 -> 상하 반전 일부 허용 (선체 표면은 방향성 약함)
fliplr: 0.5      # 기본과 동일 -> 좌우 반전
mosaic: 1.0      # 기본과 동일 -> 4장 조합 모자이크 유지
mixup: 0.15      # 기본 0.0 -> 이미지 블렌딩 활성화 (데이터 부족 보완)
copy_paste: 0.1  # 기본 0.0 -> 객체 복사-붙여넣기 소량 활성화

In [ ]:
# corrosion 데이터셋 재학습 (augmentation 강화 버전)
# torch.load weights_only 호환성 문제를 monkeypatch로 우회

import functools
import torch

_orig_load = torch.load
torch.load = functools.partial(_orig_load, weights_only=False)

import runpy
import sys

sys.argv = [
    "train.py",
    "--img", "320",
    "--batch", "16",
    "--epochs", "150",
    "--data", "/content/drive/MyDrive/hull_crack/data.yaml",
    "--weights", "yolov5n.pt",
    "--hyp", "/content/drive/MyDrive/hull_crack/hyp_corrosion_strong_aug.yaml",
    "--patience", "30",
    "--project", "/content/drive/MyDrive/hull_crack/runs",
    "--name", "corrosion_detect_v3_aug",
]

runpy.run_path("train.py", run_name="__main__")

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  with torch.cuda.amp.autocast(amp):
    80/149    0.598G   0.06162   0.04197         0       169       320:  88%|████████▊ | 29/33 [00:12<00:01,  2.11it/s]train.py:308: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
    80/149    0.598G   0.06164   0.04191         0       139       320:  91%|█████████ | 30/33 [00:12<00:01,  2.60it/s]train.py:308: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
    80/149    0.598G   0.06157   0.04218         0       145       320:  94%|█████████▍| 31/33 [00:13<00:00,  2.33it/s]train.py:308: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
    80/149    0.598G   0.06135    0.0418        

{'__name__': '__main__',
 '__doc__': "\nTrain a YOLOv5 model on a custom dataset.\n\nModels and datasets download automatically from the latest YOLOv5 release.\nModels: https://github.com/ultralytics/yolov5/tree/master/models\nDatasets: https://github.com/ultralytics/yolov5/tree/master/data\nTutorial: https://github.com/ultralytics/yolov5/wiki/Train-Custom-Data\n\nUsage:\n    $ python path/to/train.py --data coco128.yaml --weights yolov5s.pt --img 640  # from pretrained (RECOMMENDED)\n    $ python path/to/train.py --data coco128.yaml --weights '' --cfg yolov5s.yaml --img 640  # from scratch\n",
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'train.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not

In [ ]:
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 16.8 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.1
    Uninstalling protobuf-3.20.1:
      Successfully uninstalled protobuf-3.20.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you 

In [ ]:
import functools
import torch

_orig_load = torch.load
torch.load = functools.partial(_orig_load, weights_only=False)

import runpy
import sys

sys.argv = [
    "export.py",
    "--weights", "/content/drive/MyDrive/hull_crack/runs/corrosion_detect_v3_aug/weights/best.pt",
    "--img", "320",
    "--include", "onnx",
    "--opset", "12",
]

runpy.run_path("export.py", run_name="__main__")

export: data=data/coco128.yaml, weights=['/content/drive/MyDrive/hull_crack/runs/corrosion_detect_v3_aug/weights/best.pt'], imgsz=[320], batch_size=1, device=cpu, half=False, inplace=False, train=False, keras=False, optimize=False, int8=False, dynamic=False, simplify=False, opset=12, verbose=False, workspace=4, nms=False, agnostic_nms=False, topk_per_class=100, topk_all=100, iou_thres=0.45, conf_thres=0.25, include=['onnx']
YOLOv5 🚀 v6.2-0-gd3ea0df8 Python-3.12.13 torch-2.11.0+cu128 CPU

Fusing layers... 
Model summary: 213 layers, 1760518 parameters, 0 gradients, 4.1 GFLOPs

PyTorch: starting from /content/drive/MyDrive/hull_crack/runs/corrosion_detect_v3_aug/weights/best.pt with output shape (1, 6300, 6) (3.6 MB)

ONNX: starting export with onnx 1.22.0...
W0810 01:28:56.934000 1601 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 12 is a lower version than we have implementations for. Automatic vers

{'__name__': '__main__',
 '__doc__': '\nExport a YOLOv5 PyTorch model to other formats. TensorFlow exports authored by https://github.com/zldrobit\n\nFormat                      | `export.py --include`         | Model\n---                         | ---                           | ---\nPyTorch                     | -                             | yolov5s.pt\nTorchScript                 | `torchscript`                 | yolov5s.torchscript\nONNX                        | `onnx`                        | yolov5s.onnx\nOpenVINO                    | `openvino`                    | yolov5s_openvino_model/\nTensorRT                    | `engine`                      | yolov5s.engine\nCoreML                      | `coreml`                      | yolov5s.mlmodel\nTensorFlow SavedModel       | `saved_model`                 | yolov5s_saved_model/\nTensorFlow GraphDef         | `pb`                          | yolov5s.pb\nTensorFlow Lite             | `tflite`                      | yolov5s.tflite\nT

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
from collections import Counter

lbl_dir = "/content/drive/MyDrive/hull_crack/train_data/train/labels_steel"

class_counter = Counter()
file_count = 0
for fname in os.listdir(lbl_dir):
    if fname.endswith(".txt"):
        file_count += 1
        with open(os.path.join(lbl_dir, fname)) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    class_counter[parts[0]] += 1

print(f"라벨 파일 총 개수: {file_count}")
print(f"class 분포: {class_counter}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
라벨 파일 총 개수: 268
class 분포: Counter({'1': 2132})


In [ ]:
# corrosion 단독 데이터셋 통합 스크립트
# - 기존 410장(class 85, corrosion)과 신규 268장(class 1, corrosion)을 class 0으로 통일
# - crack(class 0, 원래 111장)은 이번 스코프에서 제외
# - 두 이미지/라벨 폴더를 하나의 corrosion 전용 폴더로 병합

import os
import shutil
from pathlib import Path

BASE = Path("/content/drive/MyDrive/hull_crack/train_data/train")

# 원본 위치
old_img_dir = BASE / "images"        # crack(0) + corrosion(85) 섞여있던 원본
old_lbl_dir = BASE / "labels"
new_img_dir = BASE / "images_steel"  # corrosion(1) 신규 268장
new_lbl_dir = BASE / "labels_steel"

# 최종 통합 결과 저장 위치
merged_img_dir = BASE / "images_corrosion_final"
merged_lbl_dir = BASE / "labels_corrosion_final"
merged_img_dir.mkdir(exist_ok=True)
merged_lbl_dir.mkdir(exist_ok=True)

IMAGE_EXTS = {".jpg", ".jpeg", ".JPG", ".JPEG"}


def find_image(stem, img_dir):
    for ext in IMAGE_EXTS:
        candidate = img_dir / f"{stem}{ext}"
        if candidate.exists():
            return candidate
    return None


def process_source(lbl_dir, img_dir, source_class_id, prefix):
    """source_class_id를 가진 라벨만 골라 class 0으로 바꿔서 병합 폴더에 복사"""
    copied = 0
    skipped_no_image = 0
    skipped_wrong_class = 0

    for lbl_file in lbl_dir.glob("*.txt"):
        stem = lbl_file.stem
        img_path = find_image(stem, img_dir)
        if img_path is None:
            skipped_no_image += 1
            continue

        with open(lbl_file) as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            if parts[0] == source_class_id:
                parts[0] = "0"  # corrosion -> class 0으로 통일
                new_lines.append(" ".join(parts))

        if not new_lines:
            skipped_wrong_class += 1
            continue

        # 이름 충돌 방지를 위해 prefix 붙여서 저장
        new_stem = f"{prefix}_{stem}"
        new_img_path = merged_img_dir / f"{new_stem}{img_path.suffix}"
        new_lbl_path = merged_lbl_dir / f"{new_stem}.txt"

        shutil.copy(img_path, new_img_path)
        with open(new_lbl_path, "w") as f:
            f.write("\n".join(new_lines) + "\n")

        copied += 1

    print(f"[{prefix}] 복사됨: {copied}, 이미지 없어서 스킵: {skipped_no_image}, 해당 클래스 없어서 스킵: {skipped_wrong_class}")


# 기존 410장 소스에서 class 85(corrosion)만 추출
process_source(old_lbl_dir, old_img_dir, source_class_id="85", prefix="src1")

# 신규 268장 소스에서 class 1(corrosion)만 추출
process_source(new_lbl_dir, new_img_dir, source_class_id="1", prefix="src2")

total_images = len(list(merged_img_dir.glob("*")))
total_labels = len(list(merged_lbl_dir.glob("*.txt")))
print(f"\n=== 통합 완료 ===")
print(f"최종 이미지: {total_images}장")
print(f"최종 라벨: {total_labels}개")

[src1] 복사됨: 410, 이미지 없어서 스킵: 1, 해당 클래스 없어서 스킵: 109
[src2] 복사됨: 254, 이미지 없어서 스킵: 3, 해당 클래스 없어서 스킵: 11

=== 통합 완료 ===
최종 이미지: 664장
최종 라벨: 664개


In [ ]:
# corrosion 단독 데이터셋 최종 통합 스크립트
# - 기존 train/val(class 85)과 신규 steel(class 1)을 전부 class 0으로 통일
# - 하나의 풀로 합친 뒤 80:20으로 train/val 재분할

import os
import shutil
import random
from pathlib import Path

random.seed(42)  # 재현 가능하게 고정

BASE = Path("/content/drive/MyDrive/hull_crack/train_data")

# 원본 위치들
SOURCES = [
    {"img": BASE / "train" / "images", "lbl": BASE / "train" / "labels", "class_id": "85", "prefix": "orig_train"},
    {"img": BASE / "val" / "images",   "lbl": BASE / "val" / "labels",   "class_id": "85", "prefix": "orig_val"},
    {"img": BASE / "train" / "images_steel", "lbl": BASE / "train" / "labels_steel", "class_id": "1", "prefix": "steel"},
]

# 임시 통합 풀 (분할 전 전체 모음)
POOL_IMG = BASE / "corrosion_pool" / "images"
POOL_LBL = BASE / "corrosion_pool" / "labels"
POOL_IMG.mkdir(parents=True, exist_ok=True)
POOL_LBL.mkdir(parents=True, exist_ok=True)

# 최종 출력 위치
FINAL_TRAIN_IMG = BASE / "corrosion_final" / "train" / "images"
FINAL_TRAIN_LBL = BASE / "corrosion_final" / "train" / "labels"
FINAL_VAL_IMG = BASE / "corrosion_final" / "val" / "images"
FINAL_VAL_LBL = BASE / "corrosion_final" / "val" / "labels"
for d in [FINAL_TRAIN_IMG, FINAL_TRAIN_LBL, FINAL_VAL_IMG, FINAL_VAL_LBL]:
    d.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".jpg", ".jpeg", ".JPG", ".JPEG"}


def find_image(stem, img_dir):
    for ext in IMAGE_EXTS:
        candidate = img_dir / f"{stem}{ext}"
        if candidate.exists():
            return candidate
    return None


# ===== 1단계: 각 소스에서 corrosion만 추출해서 풀로 모으기 =====
total_pooled = 0
for src in SOURCES:
    if not src["lbl"].exists():
        print(f"[SKIP] {src['lbl']} 없음")
        continue

    copied = 0
    for lbl_file in src["lbl"].glob("*.txt"):
        stem = lbl_file.stem
        img_path = find_image(stem, src["img"])
        if img_path is None:
            continue

        with open(lbl_file) as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if parts and parts[0] == src["class_id"]:
                parts[0] = "0"
                new_lines.append(" ".join(parts))

        if not new_lines:
            continue

        new_stem = f"{src['prefix']}_{stem}"
        shutil.copy(img_path, POOL_IMG / f"{new_stem}{img_path.suffix}")
        with open(POOL_LBL / f"{new_stem}.txt", "w") as f:
            f.write("\n".join(new_lines) + "\n")
        copied += 1

    print(f"[{src['prefix']}] {copied}장 풀로 이동")
    total_pooled += copied

print(f"\n총 풀에 모인 corrosion 이미지: {total_pooled}장")

# ===== 2단계: 풀 전체를 80:20으로 train/val 재분할 =====
all_stems = [p.stem for p in POOL_IMG.glob("*") if p.suffix in IMAGE_EXTS]
random.shuffle(all_stems)

split_idx = int(len(all_stems) * 0.8)
train_stems = all_stems[:split_idx]
val_stems = all_stems[split_idx:]


def move_split(stems, dest_img, dest_lbl):
    for stem in stems:
        img_path = find_image(stem, POOL_IMG)
        lbl_path = POOL_LBL / f"{stem}.txt"
        if img_path and lbl_path.exists():
            shutil.copy(img_path, dest_img / img_path.name)
            shutil.copy(lbl_path, dest_lbl / lbl_path.name)


move_split(train_stems, FINAL_TRAIN_IMG, FINAL_TRAIN_LBL)
move_split(val_stems, FINAL_VAL_IMG, FINAL_VAL_LBL)

print(f"\n=== 최종 분할 완료 ===")
print(f"train: {len(train_stems)}장")
print(f"val: {len(val_stems)}장")
print(f"\ndata.yaml에 아래 경로 사용:")
print(f"path: {BASE / 'corrosion_final'}")
print(f"train: train/images")
print(f"val: val/images")
print(f"nc: 1")
print(f"names: ['corrosion']")

[orig_train] 410장 풀로 이동
[orig_val] 0장 풀로 이동
[steel] 254장 풀로 이동

총 풀에 모인 corrosion 이미지: 664장

=== 최종 분할 완료 ===
train: 531장
val: 133장

data.yaml에 아래 경로 사용:
path: /content/drive/MyDrive/hull_crack/train_data/corrosion_final
train: train/images
val: val/images
nc: 1
names: ['corrosion']


In [ ]:
data_yaml_content = """path: /content/drive/MyDrive/hull_crack/train_data/corrosion_final
train: train/images
val: val/images

nc: 1
names: ['corrosion']
"""

with open("/content/drive/MyDrive/hull_crack/data.yaml", "w") as f:
    f.write(data_yaml_content)

print("data.yaml 교체 완료")
with open("/content/drive/MyDrive/hull_crack/data.yaml") as f:
    print(f.read())

data.yaml 교체 완료
path: /content/drive/MyDrive/hull_crack/train_data/corrosion_final
train: train/images
val: val/images

nc: 1
names: ['corrosion']



In [ ]:
# corrosion 데이터셋 학습 실행 스크립트
# torch.load weights_only 호환성 문제를 monkeypatch로 우회

import functools
import torch

_orig_load = torch.load
torch.load = functools.partial(_orig_load, weights_only=False)

import runpy
import sys

sys.argv = [
    "train.py",
    "--img", "320",
    "--batch", "16",
    "--epochs", "100",
    "--data", "/content/drive/MyDrive/hull_crack/data.yaml",
    "--weights", "yolov5n.pt",
    "--patience", "20",
    "--project", "/content/drive/MyDrive/hull_crack/runs",
    "--name", "corrosion_detect_v1",
]

runpy.run_path("train.py", run_name="__main__")

FileNotFoundError: [Errno 2] No such file or directory: '/content/train.py'

In [ ]:
# 학습된 corrosion 모델 테스트 스크립트
# 1) val 데이터셋 전체에 대한 정량 평가 (mAP, Precision, Recall)
# 2) 샘플 이미지 몇 장에 대한 시각적 추론 결과 확인

import functools
import torch

_orig_load = torch.load
torch.load = functools.partial(_orig_load, weights_only=False)

import runpy
import sys

# ===== 1) 정량 평가: val.py로 mAP/Precision/Recall 계산 =====
BEST_WEIGHTS = "/content/drive/MyDrive/hull_crack/runs/corrosion_detect_v1/weights/best.pt"

sys.argv = [
    "val.py",
    "--data", "/content/drive/MyDrive/hull_crack/data.yaml",
    "--weights", BEST_WEIGHTS,
    "--img", "320",
    "--task", "val",
    "--project", "/content/drive/MyDrive/hull_crack/runs",
    "--name", "corrosion_eval_v1",
]

runpy.run_path("val.py", run_name="__main__")

In [ ]:
# 학습된 corrosion 모델로 샘플 이미지 몇 장에 대해 시각적 추론 결과 확인
# detect.py를 사용해 결과 이미지를 저장하고, 노트북에서 바로 시각화

import functools
import torch

_orig_load = torch.load
torch.load = functools.partial(_orig_load, weights_only=False)

import runpy
import sys

BEST_WEIGHTS = "/content/drive/MyDrive/hull_crack/runs/corrosion_detect_v1/weights/best.pt"
# val 이미지 폴더 전체에 대해 추론 (원하는 이미지 폴더로 바꿔도 됨)
SOURCE_DIR = "/content/drive/MyDrive/hull_crack/train_data/corrosion_final/val/images"

sys.argv = [
    "detect.py",
    "--weights", BEST_WEIGHTS,
    "--source", SOURCE_DIR,
    "--img", "320",
    "--conf-thres", "0.25",
    "--project", "/content/drive/MyDrive/hull_crack/runs",
    "--name", "corrosion_detect_test",
]

runpy.run_path("detect.py", run_name="__main__")

# ===== 결과 이미지 몇 장 노트북에서 바로 확인 =====
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

result_dir = "/content/drive/MyDrive/hull_crack/runs/corrosion_detect_test"
result_images = sorted(glob.glob(f"{result_dir}/*.jpg"))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, img_path in zip(axes.flat, result_images):
    img = mpimg.imread(img_path)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(img_path.split('/')[-1], fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
!nvidia-smi